# The Framework Landscape: A Concrete Comparison [Agent Patterns - Module 13]

> **MLCourse - Agentic AI - Agent Patterns**

Framework comparisons are usually written as feature tables by people
selling something. This one is written as **code that runs**: the same
writer-and-critic task, on the same Groq model, implemented in AutoGen and in
LangGraph in this notebook, and pointed at the working CrewAI implementation
already in this repository.

### What you will learn

1. The same task in AutoGen and LangGraph, both executed here.
2. Where CrewAI expresses it, in this course's own code.
3. What each framework makes easy, and what it hides.
4. How to choose - and when the answer is "none of them".
5. Semantic Kernel and the rest of the landscape, honestly scoped.

### Key takeaways

- The frameworks differ mainly in **who owns the control flow**.
- Every abstraction that saves you code also hides a failure mode.
- "No framework" is a legitimate and often correct answer.

### Setup: imports, environment, track discovery


In [ ]:
import os
import sys
import json
import time
import random
import asyncio
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq's OpenAI-compatible endpoint)")


### Point AutoGen at Groq


In [ ]:
# AutoGen ships an OpenAI client. Groq exposes an OpenAI-COMPATIBLE endpoint,
# so we reuse that client and only change the base_url. This is the standard
# way to run AutoGen on a non-OpenAI provider - there is no Groq-specific
# client to install.
#
# `model_info` is REQUIRED for any model AutoGen does not have a built-in
# capability table for. Without it you get a ValueError before a single
# request goes out. You are telling the framework what the model can do.

from autogen_ext.models.openai import OpenAIChatCompletionClient

def make_client():
    return OpenAIChatCompletionClient(
        model=MODEL,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1",   # <- the only Groq-specific line
        temperature=0.0,
        max_tokens=500,                              # free tier is 8000 TPM
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": "unknown",
            "structured_output": False,
        },
    )

print("make_client() ready")


### The shared task


In [ ]:
# This module solves ONE task in three frameworks so the comparison is about
# the frameworks, not the problem. It is the same shape as the CrewAI crew in
# 04_crewai/01_fundamentals/05_research_assistant_crew: a writer produces a
# short piece from fixed reference notes, a critic reviews it, the writer
# revises. Deliberately tiny - we are studying plumbing, not prose.

NOTES = """Research notes: agent frameworks, 2026.
- Frameworks matured: CrewAI (role-based crews), LangGraph (explicit graphs),
  AutoGen (conversational agents).
- Agents are moving from demos to production.
- Main challenges: reliability, cost control, observability."""

TASK = ("Using ONLY the notes below, write a 2-sentence summary for an "
        "engineering newsletter.\n\n" + NOTES)

print(TASK)


### 1. AutoGen: agents in a conversation

Control flow is **implicit**. You choose participants and a stopping rule;
the framework runs the turn order. You do not describe the path - you
describe the room and the exit condition.

### AutoGen implementation


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

client = make_client()

ag_writer = AssistantAgent(
    "writer", model_client=client,
    system_message=("Write concise newsletter copy from the given facts. "
                    "On feedback, output the full revised text."))
ag_critic = AssistantAgent(
    "critic", model_client=client,
    system_message=("Give at most one specific improvement in one sentence. "
                    "If the copy is accurate and under 3 sentences, reply "
                    "exactly: APPROVED"))

ag_team = RoundRobinGroupChat(
    [ag_writer, ag_critic],
    termination_condition=TextMentionTermination("APPROVED") | MaxMessageTermination(5),
)

t0 = time.time()
ag_result = await ag_team.run(task=TASK)
autogen_seconds = time.time() - t0

autogen_tokens = sum((m.models_usage.prompt_tokens + m.models_usage.completion_tokens)
                     for m in ag_result.messages if getattr(m, "models_usage", None))
autogen_calls = sum(1 for m in ag_result.messages if getattr(m, "models_usage", None))
autogen_output = ag_result.messages[-2].content if len(ag_result.messages) > 1 else ""

print("stop_reason:", ag_result.stop_reason)
print(f"messages={len(ag_result.messages)}  llm_calls={autogen_calls}  "
      f"tokens={autogen_tokens}  {autogen_seconds:.1f}s")
print()
print("final draft:\n", autogen_output)
await client.close()


### 2. LangGraph: an explicit state machine

Control flow is **explicit and visible**. You declare a state schema, write
nodes as plain functions, and wire the edges yourself - including the
conditional edge that decides "revise again or stop".

Nothing is hidden. That is the trade: more code, total visibility.

### LangGraph implementation


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL, api_key=GROQ_API_KEY, temperature=0, max_tokens=500)

class State(TypedDict):
    task: str
    draft: str
    feedback: str
    rounds: int

lg_calls = 0

def write(state: State) -> State:
    global lg_calls
    prompt = state["task"] if not state["feedback"] else (
        f"{state['task']}\n\nYour previous draft:\n{state['draft']}\n\n"
        f"Editor feedback: {state['feedback']}\n\nOutput the full revised text only.")
    lg_calls += 1
    out = llm.invoke([
        ("system", "Write concise newsletter copy from the given facts."),
        ("user", prompt)]).content
    return {"draft": out.strip(), "rounds": state["rounds"] + 1}

def review(state: State) -> State:
    global lg_calls
    lg_calls += 1
    out = llm.invoke([
        ("system", "Give at most one specific improvement in one sentence. "
                   "If the copy is accurate and under 3 sentences, reply exactly: APPROVED"),
        ("user", state["draft"])]).content
    return {"feedback": out.strip()}

def should_continue(state: State) -> str:
    # The stopping rule is a plain Python function you can unit-test.
    if "APPROVED" in state["feedback"].upper():
        return "done"
    return "done" if state["rounds"] >= 2 else "revise"

g = StateGraph(State)
g.add_node("write", write)
g.add_node("review", review)
g.add_edge(START, "write")
g.add_edge("write", "review")
g.add_conditional_edges("review", should_continue, {"revise": "write", "done": END})
graph = g.compile()

print(graph.get_graph().draw_ascii())


### Run the graph


In [ ]:
t0 = time.time()
lg_result = graph.invoke({"task": TASK, "draft": "", "feedback": "", "rounds": 0})
langgraph_seconds = time.time() - t0

print(f"rounds={lg_result['rounds']}  llm_calls={lg_calls}  {langgraph_seconds:.1f}s")
print("feedback:", lg_result["feedback"][:120])
print()
print("final draft:\n", lg_result["draft"])


### The difference, in one observation

Scroll back to the ASCII diagram. **That is the whole program.** You can read
the control flow, point at the loop, and unit-test `should_continue` without
an LLM.

The AutoGen version has no such diagram, because there is no graph - the
control flow lives in the team class and the termination condition. Whether
that is liberating or worrying depends entirely on how much you need to
explain the system to somebody else.

### 3. CrewAI: roles and a process

CrewAI takes a third position: control flow comes from **roles plus a
process**. You declare who the agents are (role, goal, backstory) and what
the tasks are, and `Process.sequential` chains them, passing each task's
output to the next through `context`.

The full working version of this exact shape is already in this repository -
a researcher, writer and editor over a reference file:

- `03_agentic_ai/04_crewai/01_fundamentals/05_research_assistant_crew/research_crew.py`
- `03_agentic_ai/04_crewai/01_fundamentals/03_tasks_and_processes/`

Its skeleton, for comparison (not executed here - it is that module's job):

```python
writer = Agent(role="Newsletter Writer",
               goal="Turn research notes into concise copy",
               backstory="You write for a technical audience.",
               llm=llm)
editor = Agent(role="Editor",
               goal="Ensure accuracy and concision",
               backstory="You cut every unnecessary word.",
               llm=llm)

draft = Task(description=TASK, expected_output="2 sentences", agent=writer)
polish = Task(description="Tighten the draft.", expected_output="2 sentences",
              agent=editor, context=[draft])

crew = Crew(agents=[writer, editor], tasks=[draft, polish],
            process=Process.sequential)
result = crew.kickoff()
```

Notice what CrewAI asks for that the others do not: **`expected_output`**.
Every task declares its own acceptance criteria. That is a genuinely good
idea, and it is the framework's most transferable lesson - most agent bugs
are unstated expectations.

Notice also what it costs: `backstory` and `goal` are prompt text you write
in English, which is exactly the hand-tuning that module 09 argued against.

### 4. The comparison table

| | LangChain | LangGraph | CrewAI | AutoGen |
|---|---|---|---|---|
| **Core abstraction** | Chain of runnables | State graph | Agents + tasks + process | Agents in a conversation |
| **Who owns control flow** | You, linearly | You, explicitly | The process | The team + termination rule |
| **State** | Passed along the chain | Declared `TypedDict` | Task context chaining | The message list |
| **Loops / cycles** | Awkward | First-class | Limited (flows) | Natural (sometimes too natural) |
| **Stopping rule** | End of chain | Conditional edge you write | End of task list | Termination condition |
| **Human in the loop** | Manual | `interrupt()` + checkpointer | Callbacks / flows | `UserProxyAgent` |
| **Durability** | None | Checkpointers | Flow persistence | Team state save/load |
| **Visualisable** | Partly | Yes, natively | No | No |
| **Prompt surface** | Your templates | Your templates | Role/goal/backstory | System messages |
| **Best at** | Straight pipelines | Complex, auditable control flow | Fast role-based team drafts | Emergent multi-agent dialogue |
| **Worst at** | Anything cyclic | Boilerplate for simple tasks | Fine-grained control | Predictable cost |

### How to actually choose

Ask two questions.

**1. Do you know the control flow in advance?**
If yes - and you almost always do - use the framework that makes it explicit.
LangGraph, or plain Python. Emergent conversation is fascinating in a demo
and hard to defend in an incident review.

**2. Do you need durability, auditability, or human approval?**
If yes, LangGraph's checkpointer model is the most developed answer in this
list, and the rest of this course leans on it for exactly that reason.

Then: **consider no framework at all.** A `while` loop, a list of messages,
and direct provider calls is a legitimate architecture. You give up
checkpointing and tool plumbing; you gain the ability to read your entire
agent in one sitting. Many production agents are exactly this. Adopt a
framework when you can name the specific thing it does for you.

### What the two runs actually cost


In [ ]:
# Not a benchmark - one run each, one tiny task. Reported so the comparison is
# grounded in numbers rather than adjectives.

print(f"{'framework':12s}{'llm calls':>11s}{'seconds':>10s}")
print("-" * 33)
print(f"{'AutoGen':12s}{autogen_calls:>11d}{autogen_seconds:>10.1f}")
print(f"{'LangGraph':12s}{lg_calls:>11d}{langgraph_seconds:>10.1f}")
print("-" * 33)
print()
print(f"AutoGen token total (reported by the framework): {autogen_tokens}")
print()
print("Two caveats worth stating out loud:")
print("  1. n=1. Different runs converge in different numbers of turns.")
print("  2. Both produced acceptable copy. On a task this small, framework")
print("     choice affects the CODE far more than the OUTPUT.")


### 5. The rest of the landscape

**Semantic Kernel** (Microsoft). An SDK built around *plugins* (functions the
model can invoke) and *planners*, with first-class C# and Python support and
strong Azure integration. Its centre of gravity is enterprise .NET.

It is covered here **in prose only, deliberately**: its Python connector
story is centred on Azure OpenAI and OpenAI, and this course is Groq-only.
Rather than ship a notebook resting on an untested provider path, the honest
position is to describe the model and point you at the docs. If you work in a
Microsoft shop, it is the most natural of these options; if you do not, its
abstractions will feel like they are solving somebody else's problem.

**OpenAI Agents SDK.** Excluded from this course by design - it requires
OpenAI as the provider, and every module here runs on Groq. Worth knowing it
exists; the concepts (handoffs, guardrails, sessions) map onto what you have
already built.

**Others you will see named:** `smolagents` (HuggingFace, code-writing agents,
very small), `Pydantic AI` (type-first, excellent validation story), `Agno`
and `Atomic Agents` (lightweight), `LlamaIndex Workflows` (event-driven,
RAG-adjacent).

The field churns fast. **The abstractions are more durable than the
libraries** - graphs, roles, conversations, termination, checkpoints,
handoffs - which is why this course teaches those and treats the imports as
an implementation detail.

### Module wrap-up

1. **AutoGen basics** - Groq via `base_url`, the mandatory `model_info`,
   async everywhere, and the message list as state.
2. **Conversation** - teams, turn-taking, and termination as the single most
   important design decision, with the quadratic prompt cost measured.
3. **Tools** - a plain function is a tool, the typed tool transcript, and
   `reflect_on_tool_use` as a real cost lever.
4. **Comparison** - the same task in AutoGen and LangGraph, side by side with
   CrewAI's expression of it, and an honest table.

The one sentence to keep: **frameworks differ mainly in who owns the control
flow, and the right answer is usually "you".**

### Related modules

- `02_langgraph/06_multi_agent_systems` - supervisor, swarm and parallel
  patterns, the graph-native version of this notebook's teams.
- `04_crewai/01_fundamentals/05_research_assistant_crew` - the CrewAI
  implementation referenced above.
- `06_agent_patterns/06_multi_agent_debate` - when multiple agents actually
  beat one, and when they just cost 3x.
- `06_agent_patterns/09_prompt_optimization` - the argument against
  hand-written `backstory` strings.